# Liquid Clusteringハンズオン学習ノートブック
[Liquid Clusteringを動かして理解する](https://qiita.com/taka_yayoi/items/0c66f5c0784860652b75)

このノートブックでは、Delta LakeのLiquid Clustering(LC)を  
実際にコードを動かしながら学びます。

特に以下の疑問に答えることを目的としています:
- LCを有効にしたら、OPTIMIZEは不要になるのか?
- LCを有効にしたら、VACUUMは不要になるのか?
- そもそもLCで何が変わるのか?

**対象読者**: Delta Lakeの基本(Parquetファイル、トランザクションログ、OPTIMIZE、VACUUM)は理解しているが、LCは初めての方

**前提条件**
- Databricks Runtime 15.4 LTS以上
- Unity Catalog環境(一部機能はUC管理テーブルが必要)

**参考ドキュメント**
- [テーブルにLiquid Clusteringを使用する](https://learn.microsoft.com/ja-jp/azure/databricks/delta/clustering)(Azure)
- [Predictive Optimization](https://docs.databricks.com/aws/en/optimizations/predictive-optimization)

# Part 1: Liquid Clusteringとは何か

## Data Skippingの仕組み

LCを理解するには、まずDelta Lakeの**data skipping**の仕組みを知る必要があります。

Delta Lakeでは、各Parquetファイルがテーブルに追加されるとき、  
ファイル内の各カラムについて**min値(最小値)とmax値(最大値)**が  
Delta Log(トランザクションログ)に記録されます。

クエリに`WHERE region = 'APAC'`というフィルタがある場合、  
Delta Lakeは実際にファイルを読む前に、Delta Logに記録されたmin/maxを確認します。

- file_00のregion: min='APAC', max='NA' → APACを含む可能性あり → **読む**
- file_01のregion: min='EMEA', max='LATAM' → APACを含む可能性なし → **スキップ**

これがdata skippingです。  
フィルタ条件に合わないファイルを読み飛ばすことで、クエリが高速化されます。

## data skippingの効果はデータの並び方に依存する

ここで重要なのは、data skippingの効果は**ファイル内のデータの並び方**に依存することです。

例えば、あるテーブルに4つのリージョン(APAC, EMEA, LATAM, NA)のデータがあるとします。

**クラスタリングなし**: 各ファイルに4リージョンすべてが混在  
→ 全ファイルのmin='APAC', max='NA' → どんなフィルタでも全ファイルを読む必要あり

**クラスタリングあり**: 同じリージョンのデータが同じファイルに集約  
→ file_00はAPACのみ(min='APAC', max='APAC')、file_01はEMEAのみ、...  
→ `WHERE region = 'APAC'`でfile_00だけ読めばよい(残りはスキップ)

つまり、**同じキー値のデータを同じファイルに集めることで、  
各ファイルのmin/max範囲が狭くなり、スキップできるファイルが増える**。  
これがクラスタリングの効果の本質です。

## データの並び方を制御する3つの方法の比較

| 観点 | パーティショニング | ZORDER | Liquid Clustering |
|---|---|---|---|
| 仕組み | ディレクトリ分割 | OPTIMIZE時に並べ替え | OPTIMIZE時に並べ替え |
| キーの変更 | テーブル再作成が必要 | OPTIMIZE時に都度指定 | `ALTER TABLE`で即変更 |
| キーの数 | 1〜2が限界 | 実質制限なし | 最大4 |
| 高カーディナリティ | 小さいファイルが大量発生 | 問題なし | 問題なし |
| OPTIMIZEの処理方式 | コンパクションのみ | 毎回全ファイル再書込 | **増分処理**(未処理分のみ) |
| VACUUMの要否 | 必要 | 必要 | 必要 |
| 併用 | ZORDERと併用可 | パーティションと併用可 | **どちらとも併用不可** |

LCはパーティショニングとZORDERの**置き換え**として設計されています。  
Databricksは**すべての新規テーブルでLCの使用を推奨**しています。

**参考**
- [Data skipping](https://learn.microsoft.com/ja-jp/azure/databricks/delta/data-skipping)
- [Liquid Clustering](https://learn.microsoft.com/ja-jp/azure/databricks/delta/clustering)
- [OPTIMIZE](https://learn.microsoft.com/ja-jp/azure/databricks/sql/language-manual/delta-optimize)

# Part 2: セットアップ

In [0]:
%run ./00_config

# Part 3: 可視化ヘルパー関数の定義

**まずこのセルを実行してください。**

| 関数 | 何がわかるか |
|---|---|
| `snapshot(table, col)` | 現在のファイル状態を記録する(OPTIMIZE前後の比較用) |
| `show_optimize_impact(before, after)` | OPTIMIZE前後で「何が変わり、結果どうなったか」をストーリーとして表示 |
| `track_storage_changes(table)` | 各操作でファイルが何個追加され、何個削除されたかを時系列で表示 |

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def snapshot(table_name, cluster_col):
    """
    現在のテーブルのファイル状態を記録する。
    戻り値を show_optimize_impact() に渡して前後比較に使う。
    """
    detail = spark.sql(f"DESCRIBE DETAIL {table_name}").collect()[0]
    total_files = detail["numFiles"]

    file_key_df = spark.sql(f"""
        SELECT 
            _metadata.file_name as file_name,
            CAST({cluster_col} AS STRING) as key_val
        FROM {table_name}
        GROUP BY _metadata.file_name, {cluster_col}
    """).collect()

    file_keys = {}
    all_keys = set()
    for row in file_key_df:
        file_keys.setdefault(row["file_name"], set()).add(row["key_val"])
        all_keys.add(row["key_val"])
    all_keys = sorted(all_keys)

    reads = {}
    for k in all_keys:
        reads[k] = sum(1 for fkeys in file_keys.values() if k in fkeys)

    avg_distinct = sum(len(v) for v in file_keys.values()) / len(file_keys) if file_keys else 0

    return {
        "file_keys": file_keys,
        "all_keys": all_keys,
        "reads": reads,
        "total_files": total_files,
        "avg_distinct": avg_distinct,
        "col": cluster_col,
        "table": table_name,
    }


def show_optimize_impact(before, after):
    """
    OPTIMIZE前後のsnapshotを受け取り、
    「何が変わり、結果としてスキャン効率がどう改善したか」をストーリーとして表示する。

    1. テキストで因果関係(再編成 → ファイルの中身の変化 → クエリへの効果)を説明
    2. ヒートマップでファイルの中身の変化を視覚的に表示(低カーディナリティ時)
    3. 棒グラフで読むファイル数の前後比較を表示
    """
    col = before["col"]
    b_total = before["total_files"]
    a_total = after["total_files"]
    b_avg_d = before["avg_distinct"]
    a_avg_d = after["avg_distinct"]
    all_keys = before["all_keys"]

    # OPTIMIZEメトリクスを取得
    history = spark.sql(f"DESCRIBE HISTORY {after['table']}")
    opt_row = (history
        .filter(F.col("operation") == "OPTIMIZE")
        .orderBy(F.col("version").desc())
        .first())
    opt_metrics = opt_row["operationMetrics"] if opt_row and opt_row["operationMetrics"] else {}
    opt_added = int(opt_metrics.get("numAddedFiles", 0))
    opt_removed = int(opt_metrics.get("numRemovedFiles", 0))

    # --- テキストストーリー ---
    print(f"{'='*70}")
    print(f" OPTIMIZEの効果 ({col})")
    print(f"{'='*70}")
    print()
    print(f" [1. 再編成]")
    print(f"   OPTIMIZEが{opt_removed}個のファイルを読み込み、")
    print(f"   {col}キーでデータを並べ替えて、{opt_added}個の新ファイルに書き直した。")
    print()
    print(f" [2. ファイルの中身の変化]")
    print(f"   OPTIMIZE前: 1ファイルあたり平均 {b_avg_d:.1f} 種類の{col}値を含んでいた")
    print(f"   OPTIMIZE後: 1ファイルあたり平均 {a_avg_d:.1f} 種類の{col}値を含む")
    print()
    print(f" [3. クエリへの効果]")
    print(f"   WHERE {col} = ... で検索したとき、読むファイル数がどう変わったか:")
    print()

    display_keys = all_keys
    if len(all_keys) > 10:
        step = max(1, len(all_keys) // 8)
        display_keys = all_keys[::step][:8]
        print(f"   ({len(all_keys)}種類の値のうち代表的な値を表示)")
        print()

    for k in display_keys:
        b_reads = before["reads"].get(k, b_total)
        a_reads = after["reads"].get(k, a_total)
        b_skip = (1 - b_reads / b_total) * 100 if b_total > 0 else 0
        a_skip = (1 - a_reads / a_total) * 100 if a_total > 0 else 0
        reduction = b_reads - a_reads
        print(f"   = '{k}': {b_reads}/{b_total}ファイル → {a_reads}/{a_total}ファイル")
        print(f"     スキップ {b_skip:.0f}% → {a_skip:.0f}%  ({reduction}ファイル削減)")

    b_avg_reads = sum(before["reads"].values()) / len(before["reads"]) if before["reads"] else 0
    a_avg_reads = sum(after["reads"].values()) / len(after["reads"]) if after["reads"] else 0
    b_avg_skip = (1 - b_avg_reads / b_total) * 100 if b_total > 0 else 0
    a_avg_skip = (1 - a_avg_reads / a_total) * 100 if a_total > 0 else 0

    print()
    print(f" [全体平均]")
    print(f"   読むファイル数: {b_avg_reads:.0f} → {a_avg_reads:.0f}")
    print(f"   スキップ率: {b_avg_skip:.0f}% → {a_avg_skip:.0f}%")
    print(f"{'='*70}")

    # --- ヒートマップ(低カーディナリティ時のみ) ---
    # 「各ファイルがどのキー値のデータを含んでいるか」を前後で比較する。
    # 赤いセル = そのファイルにそのキー値が含まれる
    # 空白 = 含まれない(スキップ可能)
    # OPTIMIZE前は赤だらけ → OPTIMIZE後は赤がまとまる、という変化が一目でわかる。
    if len(all_keys) <= 20:
        MAX_FILES = 12
        before_files = before["file_keys"]
        after_files = after["file_keys"]

        # ファイルが多い場合は均等にサンプリング
        b_names = sorted(before_files.keys())
        a_names = sorted(after_files.keys())
        if len(b_names) > MAX_FILES:
            step = len(b_names) // MAX_FILES
            b_names = b_names[::step][:MAX_FILES]
        if len(a_names) > MAX_FILES:
            step = len(a_names) // MAX_FILES
            a_names = a_names[::step][:MAX_FILES]

        b_labels = [f"f{i:02d}" for i in range(len(b_names))]
        a_labels = [f"f{i:02d}" for i in range(len(a_names))]

        b_matrix = [[1 if k in before_files[f] else 0 for k in all_keys] for f in b_names]
        a_matrix = [[1 if k in after_files[f] else 0 for k in all_keys] for f in a_names]

        max_rows = max(len(b_names), len(a_names))
        colorscale = [[0, "#F1EFE8"], [1, "#E24B4A"]]

        sampled_note = ""
        if b_total > MAX_FILES or a_total > MAX_FILES:
            sampled_note = f" (全ファイルから{MAX_FILES}件をサンプル表示)"

        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=[
                f"OPTIMIZE前 ({b_total}ファイル)",
                f"OPTIMIZE後 ({a_total}ファイル)",
            ],
            horizontal_spacing=0.12,
        )
        fig.add_trace(go.Heatmap(
            z=b_matrix, x=all_keys, y=b_labels,
            colorscale=colorscale, showscale=False, xgap=3, ygap=2,
            hovertemplate="ファイル: %{y}<br>キー値: %{x}<br>含む: %{z}<extra></extra>",
        ), row=1, col=1)
        fig.add_trace(go.Heatmap(
            z=a_matrix, x=all_keys, y=a_labels,
            colorscale=colorscale, showscale=False, xgap=3, ygap=2,
            hovertemplate="ファイル: %{y}<br>キー値: %{x}<br>含む: %{z}<extra></extra>",
        ), row=1, col=2)
        fig.update_layout(
            title=(
                f"各ファイルが含む{col}値の分布{sampled_note}<br>"
                f"<sup>赤 = そのファイルにその{col}値が含まれる | 空白 = 含まれない(スキップ可能)</sup>"
            ),
            height=max(300, max_rows * 28 + 150),
        )
        fig.show()

    # --- 棒グラフ: 読むファイル数の前後比較 ---
    if len(display_keys) <= 20:
        fig2 = go.Figure()
        fig2.add_trace(go.Bar(
            x=display_keys,
            y=[before["reads"].get(k, b_total) for k in display_keys],
            name=f"OPTIMIZE前 (全{b_total}ファイル)",
            marker_color="#E24B4A",
            text=[f"{before['reads'].get(k, b_total)}" for k in display_keys],
            textposition="auto",
        ))
        fig2.add_trace(go.Bar(
            x=display_keys,
            y=[after["reads"].get(k, a_total) for k in display_keys],
            name=f"OPTIMIZE後 (全{a_total}ファイル)",
            marker_color="#378ADD",
            text=[f"{after['reads'].get(k, a_total)}" for k in display_keys],
            textposition="auto",
        ))
        fig2.update_layout(
            title=f"WHERE {col} = ... で検索したとき読むファイル数の変化",
            xaxis_title=f"{col}の値",
            yaxis_title="読むファイル数(少ないほど良い)",
            barmode="group",
            height=350,
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        )
        fig2.show()


def track_storage_changes(table_name):
    """
    各操作でファイルが何個追加され、何個削除されたかを時系列で表示する。

    - 追加(青): 新しく作成されたファイル数
    - 論理削除(オレンジ): Delta Logから参照を外されたファイル数(ストレージにはまだ残っている)
    - VACUUM物理削除(赤): VACUUMでストレージから実際に消されたファイル数
    """
    history = spark.sql(f"DESCRIBE HISTORY {table_name}")
    rows = history.select("version", "operation", "operationMetrics").orderBy("version").collect()

    versions, added_list, removed_list, vac_list = [], [], [], []
    for row in rows:
        metrics = row["operationMetrics"] or {}
        op = row["operation"]
        if op == "WRITE":
            added, removed, vac = int(metrics.get("numFiles", 0)), 0, 0
        elif op == "OPTIMIZE":
            added = int(metrics.get("numAddedFiles", 0))
            removed = int(metrics.get("numRemovedFiles", 0))
            vac = 0
        elif op == "VACUUM END":
            added, removed = 0, 0
            vac = int(metrics.get("numDeletedFiles", 0))
        else:
            added, removed, vac = 0, 0, 0
        versions.append(f"v{row['version']} {op}")
        added_list.append(added)
        removed_list.append(removed)
        vac_list.append(vac)

    fig = go.Figure()
    fig.add_trace(go.Bar(x=versions, y=added_list, name="追加",
        marker_color="#378ADD", text=[str(a) if a > 0 else "" for a in added_list], textposition="auto"))
    fig.add_trace(go.Bar(x=versions, y=[-r for r in removed_list], name="論理削除",
        marker_color="#EF9F27", text=[f"-{r}" if r > 0 else "" for r in removed_list], textposition="auto"))
    fig.add_trace(go.Bar(x=versions, y=[-v for v in vac_list], name="VACUUM物理削除",
        marker_color="#E24B4A", text=[f"-{v}" if v > 0 else "" for v in vac_list], textposition="auto"))
    fig.update_layout(
        title=f"{table_name} - 各操作でのファイル増減",
        xaxis_title="バージョン / 操作",
        yaxis_title="ファイル数(上=追加, 下=削除)",
        barmode="relative", height=350,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig.add_hline(y=0, line_width=1, line_color="gray")
    fig.show()


print("可視化ヘルパー関数の定義完了")

# Part 4: テーブルの作成とデータ投入

## なぜ「CLUSTER BYなし」でテーブルを作るのか

LCの効果を観察するには、まず「クラスタリングされていない状態」が必要です。

最初から`CLUSTER BY`を指定したテーブルにデータを書き込むと、  
ファイルがある程度整理された状態で作られることがあり、  
「OPTIMIZE前後の違い」が見えにくくなります。

そこで以下の手順を取ります:
1. **CLUSTER BYなし**でテーブルを作成
2. データを投入(各ファイルにリージョンがランダムに混在)
3. OPTIMIZE前の状態を記録
4. `ALTER TABLE ... CLUSTER BY`でLC有効化 + `OPTIMIZE`
5. OPTIMIZE前後の変化をストーリーとして確認

## なぜ`delta.targetFileSize`を変更するのか

LCの効果は「複数のファイルに分かれたデータのうち、  
不要なファイルをスキップできるようになる」ことです。  
つまり、テーブルが複数のファイルで構成されていないと効果を観察できません。

OPTIMIZEはファイルを統合するとき、`delta.targetFileSize`で指定された  
サイズを目標にします。デフォルトは約1GBです。  
今回のハンズオンデータは約6MBしかないので、  
デフォルトのままだとOPTIMIZEが全データを1ファイルにまとめてしまいます。

そこで`delta.targetFileSize`を`256kb`に設定し、  
少量データでも多数のファイルに分割されるようにします。

**これは学習用の設定です。本番環境ではデフォルトのまま使用してください。**

In [0]:
%sql
DROP TABLE IF EXISTS sales_lc;

CREATE TABLE sales_lc (
  order_id BIGINT,
  order_date DATE,
  customer_id INT,
  region STRING,
  product STRING,
  amount DOUBLE,
  quantity INT
)
TBLPROPERTIES ('delta.targetFileSize' = '256kb');

In [0]:
from pyspark.sql import functions as F

regions_array = F.array(*[F.lit(r) for r in ["APAC", "EMEA", "NA", "LATAM"]])
products_array = F.array(*[F.lit(p) for p in ["Widget-A", "Widget-B", "Gadget-X", "Gadget-Y", "Service-Z"]])

df = (
    spark.range(0, 1_000_000, numPartitions=50)
    .withColumn("order_id", F.col("id"))
    .withColumn("order_date", F.date_add(F.lit("2023-01-01"), (F.rand(seed=10) * 730).cast("int")))
    .withColumn("customer_id", (F.rand(seed=42) * 10000).cast("int"))
    .withColumn("region", F.element_at(regions_array, (F.rand(seed=77) * 4).cast("int") + 1))
    .withColumn("product", F.element_at(products_array, (F.rand(seed=88) * 5).cast("int") + 1))
    .withColumn("amount", F.round(F.rand(seed=123) * 1000, 2))
    .withColumn("quantity", (F.rand(seed=456) * 20 + 1).cast("int"))
    .drop("id")
)

df.write.mode("append").saveAsTable("sales_lc")
print("データ投入完了: 1,000,000 行")

# Part 5: クラスタリング前の状態を記録する

## なぜこれを確認するのか

OPTIMIZEの効果を理解するには、「OPTIMIZEの前後で何が変わったか」を比較する必要があります。  
ここでは`snapshot`関数で現在のファイル状態を記録しておきます。  
次のPartでOPTIMIZEを実行した後に、この記録と比較します。

In [0]:
before_region = snapshot("sales_lc", "region")
print(f"記録完了: {before_region['total_files']}ファイル, 1ファイルあたり平均{before_region['avg_distinct']:.1f}リージョン")

In [0]:
before_date = snapshot("sales_lc", "order_date")
print(f"記録完了: {before_date['total_files']}ファイル, 1ファイルあたり平均{before_date['avg_distinct']:.0f}種類の日付")

# Part 6: LCの有効化とOPTIMIZE

## Step 1: ALTER TABLEでLCを有効化する

`ALTER TABLE ... CLUSTER BY`でLCを有効化します。  
この操作は**メタデータの変更のみ**で、ファイルの物理的な再編成は行われません。

In [0]:
%sql
ALTER TABLE sales_lc CLUSTER BY (region, order_date);

## Step 2: OPTIMIZEでクラスタリングを実行する

LCテーブルでのOPTIMIZEは従来と動作が異なります。

| テーブル種別 | `OPTIMIZE`の動作 |
|---|---|
| 通常のDeltaテーブル | ファイルの統合(コンパクション)のみ。データの並び順は変わらない |
| 通常 + `ZORDER BY`指定 | コンパクション + Z-Orderで並べ替え。毎回**全ファイル**を再書き込み |
| **LCテーブル** | コンパクション + **クラスタリングキーで並べ替え**。**未処理のファイルだけ**が対象 |

ポイント:
1. LCテーブルでは`OPTIMIZE`だけでクラスタリングが行われる(`ZORDER BY`の指定は不要。指定するとエラー)
2. 前回OPTIMIZEで処理済みのファイルはスキップされる(増分処理)
3. つまりLCを有効にしても、**OPTIMIZEの実行は引き続き必要**

In [0]:
%sql
OPTIMIZE sales_lc

## Step 3: OPTIMIZEで何が変わったかを確認する

`show_optimize_impact`は、OPTIMIZE前後のsnapshotを比較して  
以下をストーリーとして表示します:

1. **再編成**: OPTIMIZEが何ファイルを読んで何ファイルに書き直したか
2. **ファイルの中身の変化**: 1ファイルあたりのキー種類数がどう変わったか
3. **クエリへの効果**: 各フィルタ条件で読むファイル数がどう減ったか

この3つは因果関係で繋がっています:  
OPTIMIZEがファイルを並べ替える(1) → ファイルの中身が整理される(2) → 読むファイルが減る(3)

テキストの後にグラフが表示されます:
- **ヒートマップ**(region等の低カーディナリティ): 各ファイルがどのキー値を含んでいるかを前後で比較。赤いセル = そのファイルにそのキー値が含まれる。OPTIMIZE前は赤だらけ → OPTIMIZE後は赤がまとまる、という変化が見えます
- **棒グラフ**: 各キー値で検索したときに読むファイル数の前後比較。赤い棒が短くなっていれば改善

In [0]:
# regionキーでの効果
after_region = snapshot("sales_lc", "region")
show_optimize_impact(before_region, after_region)

In [0]:
# order_dateキーでの効果
after_date = snapshot("sales_lc", "order_date")
show_optimize_impact(before_date, after_date)

## ファイルの増減を確認する

`track_storage_changes`は、テーブルの各操作(WRITE, OPTIMIZE, VACUUM)で  
ファイルが何個追加され、何個削除されたかを時系列で表示します。

- **青い棒(上方向)**: 追加されたファイル数
- **オレンジの棒(下方向)**: 論理削除されたファイル数(Delta Logから参照が外されたが、ストレージにはまだ残っている)

OPTIMIZEの操作では青とオレンジの両方が出ます。  
旧ファイルを読み込んで新ファイルに書き直したため、追加と削除の両方が発生しています。  
オレンジの旧ファイルはまだストレージに残っているため、VACUUMで物理削除する必要があります。

In [0]:
track_storage_changes("sales_lc")

# Part 7: VACUUMの役割

## なぜVACUUMが必要なのか

Part 6のグラフで確認したように、OPTIMIZEは:
1. 新しいファイルを**追加**(青い棒)
2. 古いファイルを**論理削除**(オレンジの棒)
3. しかし古いファイルは**ストレージ上にまだ存在**している

古いファイルがすぐに消されない理由は、Delta LakeのTime Travel機能のためです。  
`SELECT * FROM sales_lc VERSION AS OF 0`のように過去のバージョンを  
参照するとき、古いファイルが必要になります。

デフォルトでは**7日間**保持されます。  
この期間を過ぎた古いファイルを物理的に削除するのがVACUUMです。

**VACUUMを定期的に実行しないと、OPTIMIZEのたびにストレージが膨張します。**  
つまり、LCを有効にしても**VACUUMは引き続き必要**です。

参考: [VACUUM](https://learn.microsoft.com/ja-jp/azure/databricks/sql/language-manual/delta-vacuum)

## VACUUMの実行

ハンズオンで即時効果を確認するため、テーブルの保持期間を一時的に0に変更します。

**この設定は本番環境では絶対に使わないでください。**  
Time Travelができなくなり、同時実行中のクエリが失敗する可能性があります。

In [0]:
%sql
-- テーブルの保持期間を0に変更(ハンズオン用)
ALTER TABLE sales_lc SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours');

In [0]:
%sql
VACUUM sales_lc

In [0]:
%sql
-- 保持期間をデフォルトに戻す
ALTER TABLE sales_lc SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 7 days');

VACUUM後の`track_storage_changes`で、赤い棒(VACUUM物理削除)が出ていれば、  
論理削除されていた旧ファイルがストレージから実際に消されたということです。

これでWRITE → OPTIMIZE → VACUUMの一連のライフサイクルが完了しました:
1. **WRITE**(青): ファイルが作られる
2. **OPTIMIZE**(青+オレンジ): 旧ファイルを読んでクラスタリングキーで並べ替え、新ファイルに再編成。旧ファイルは論理削除される
3. **VACUUM**(赤): 論理削除されていた旧ファイルをストレージから物理削除

In [0]:
track_storage_changes("sales_lc")

# Part 8: 補足トピック

以下はLCの基本を理解した上での補足事項です。

## クラスタリングキーの変更

ビジネスの変化に伴ってクエリパターンが変わったとき、  
LCでは`ALTER TABLE`一発でキーを変更できます。  
パーティショニングではテーブルの再作成が必要でした。

```sql
ALTER TABLE sales_lc CLUSTER BY (region, order_date, customer_id);
```

この操作はメタデータの変更のみです。  
次にOPTIMIZEを実行すると、新しいキーに基づいてデータが再クラスタリングされます。  
全データを一度に再クラスタリングしたい場合は`OPTIMIZE テーブル名 FULL`を使います(DBR 16.0以降)。

参考: [Liquid Clustering - クラスタリングキーの変更](https://learn.microsoft.com/ja-jp/azure/databricks/delta/clustering#change-clustering-keys)

## 自動Liquid Clustering (CLUSTER BY AUTO)

`CLUSTER BY AUTO`を使うと、Databricksがクエリの実行履歴を分析して  
最適なキーを自動的に選択・更新してくれます。  
Predictive Optimizationが有効な環境で利用可能です。

```sql
CREATE TABLE my_table (...) CLUSTER BY AUTO;
```

Predictive OptimizationはOPTIMIZE、VACUUM、ANALYZEの3つの  
メンテナンス操作も自動スケジュールします。  
つまり、CLUSTER BY AUTO + Predictive Optimizationで  
**キーの選択もメンテナンスもすべて自動化**されます。

参考: [自動Liquid Clustering](https://learn.microsoft.com/ja-jp/azure/databricks/delta/clustering#automatic-liquid-clustering) / [Predictive Optimization](https://learn.microsoft.com/ja-jp/azure/databricks/optimizations/predictive-optimization)

## 書き込み時の自動クラスタリング

LCテーブルでは、書き込みデータが一定の閾値を超えると  
書き込み時に自動でクラスタリングが行われます(clustering-on-write)。

閾値は`CLUSTER BY`で指定したキーの数によって変わります:

| CLUSTER BYの指定 | UC管理テーブルの閾値 | その他のDeltaテーブルの閾値 |
|---|---|---|
| `CLUSTER BY (col1)` — キー1つ | 64MB | 256MB |
| `CLUSTER BY (col1, col2)` — キー2つ | 256MB | 1GB |
| `CLUSTER BY (col1, col2, col3)` — キー3つ | 512MB | 2GB |
| `CLUSTER BY (col1, col2, col3, col4)` — キー4つ | 1GB | 4GB |

閾値以下の小さな書き込みでは発動しないため、**OPTIMIZEは引き続き必要**です。

参考: [Liquid Clustering - 書き込み時のクラスタリングのサイズ閾値](https://learn.microsoft.com/ja-jp/azure/databricks/delta/clustering#size-thresholds-for-clustering)

# Part 9: まとめ

## LCでもOPTIMIZEとVACUUMは必要か? → はい、両方必要

**OPTIMIZE**
- LCテーブルではOPTIMIZEがクラスタリングのトリガーになる
- `CLUSTER BY`で設定したキーに基づいてデータが並べ替えられる(`ZORDER BY`の指定は不要)
- 既にクラスタリング済みのファイルはスキップされる(増分処理)ため、ZORDERより軽い
- 手動で定期実行するか、Predictive Optimizationで自動化する

**VACUUM**
- OPTIMIZEがファイルを再編成すると、古いファイルが論理削除されてストレージに残る
- VACUUMを実行しないとストレージが膨張し続ける
- デフォルト保持期間は7日間
- 手動で定期実行するか、Predictive Optimizationで自動化する

## 運用ガイドライン

| シナリオ | OPTIMIZEの頻度 | VACUUMの頻度 |
|---|---|---|
| 頻繁なINSERT/UPDATEがあるテーブル | 1〜2時間ごと | 日次 |
| 日次バッチ処理のテーブル | バッチ後に1回 | 日次〜週次 |
| Predictive Optimizationが有効 | 自動(手動不要) | 自動(手動不要) |

# クリーンアップ

In [0]:
%sql
-- 学習用テーブルの削除(必要に応じてコメント解除)
-- DROP TABLE IF EXISTS sales_lc;
-- DROP SCHEMA IF EXISTS liquid_clustering_lab;